# Docking Aspirin into COX-1
Aspirin works by blocking **cyclooxygenase-1 (COX-1)**, an enzyme that makes pain and inflammation signals. Here we use molecular docking (AutoDock Vina) to predict how aspirin sits in the enzyme's binding channel.

**Plan**
1. Prepare the COX-1 structure (PDB 1EQG, crystallized with ibuprofen)
2. *Validate* the method by re-docking ibuprofen and measuring how close we get to the real pose
3. Dock aspirin and list the amino acids it touches
4. Compare several common remedies
5. View everything in 3D

## 1. Setup

In [ ]:
!apt-get -qq install -y openbabel > /dev/null
!pip install -q vina rdkit py3Dmol pandas matplotlib

In [ ]:
import subprocess, requests
import numpy as np, pandas as pd, matplotlib.pyplot as plt, py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from vina import Vina

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[:500])
    return r

## 2. Load COX-1 and find the binding site
The site is defined by where ibuprofen sits in the crystal structure.

In [ ]:
pdb = requests.get("https://files.rcsb.org/download/1EQG.pdb", timeout=30).text
lines = pdb.splitlines()

ibp_all = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == "IBP" and l[16] in " A"]
chain = ibp_all[0][21]
ibp = [l for l in ibp_all if l[21] == chain]
protein = [l for l in lines if l.startswith("ATOM") and l[21] == chain and l[16] in " A"]

def xyz(ls):
    return np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])] for l in ls])

center = xyz(ibp).mean(axis=0)
print("Chain used:", chain)
print("Protein atoms:", len(protein), "| Ibuprofen atoms:", len(ibp))
print("Binding site center:", center.round(2))

## 3. Prepare the receptor
We keep only the protein chain (no water, no heme) and convert it to the PDBQT format Vina needs.

In [ ]:
open("receptor.pdb", "w").write("\n".join(protein) + "\nEND\n")
sh("obabel receptor.pdb -O receptor.pdbqt -xr -h --partialcharge gasteiger")
import os; print("receptor.pdbqt ready:", os.path.exists("receptor.pdbqt"))

## 4. Docking helper functions

In [ ]:
BOX = [22, 22, 22]

def prep_ligand(smiles, name):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=7)
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToMolFile(mol, f"{name}.mol")
    sh(f"obabel {name}.mol -O {name}.pdbqt --partialcharge gasteiger")
    return f"{name}.pdbqt"

def dock(smiles, name, exhaustiveness=16):
    lig = prep_ligand(smiles, name)
    v = Vina(sf_name="vina", seed=42)
    v.set_receptor("receptor.pdbqt")
    v.set_ligand_from_file(lig)
    v.compute_vina_maps(center=center.tolist(), box_size=BOX)
    v.dock(exhaustiveness=exhaustiveness, n_poses=9)
    v.write_poses(f"{name}_out.pdbqt", n_poses=9, overwrite=True)
    sh(f"obabel {name}_out.pdbqt -O {name}_best.pdb -f 1 -l 1")
    best = v.energies()[0][0]   # kcal/mol, more negative = stronger predicted binding
    return best

def read_xyz_pdb(path):
    ls = [l for l in open(path) if l.startswith(("ATOM", "HETATM"))]
    return xyz(ls)

## 5. Validation: re-dock ibuprofen
If the method is trustworthy, the predicted pose of ibuprofen should land close to the real crystal pose. An RMSD under about 2 Å is usually considered a success.

In [ ]:
IBU = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"
ibu_score = dock(IBU, "ibuprofen")

template = Chem.MolFromSmiles(IBU)
crystal = AllChem.AssignBondOrdersFromTemplate(template, Chem.MolFromPDBBlock("\n".join(ibp), removeHs=True))
pose = AllChem.AssignBondOrdersFromTemplate(template, Chem.MolFromPDBBlock(open("ibuprofen_best.pdb").read(), removeHs=True))

try:
    rmsd = rdMolAlign.CalcRMS(pose, crystal)   # no re-alignment, symmetry aware
except Exception as e:
    rmsd = None
    print("RMSD failed:", e)

centroid_dist = np.linalg.norm(read_xyz_pdb("ibuprofen_best.pdb").mean(axis=0) - xyz(ibp).mean(axis=0))
print(f"Ibuprofen docking score: {ibu_score:.2f} kcal/mol")
print(f"Pose RMSD vs crystal: {rmsd if rmsd is None else round(rmsd, 2)} Å")
print(f"Centroid distance: {centroid_dist:.2f} Å")

## 6. Dock aspirin

In [ ]:
ASPIRIN = "CC(=O)Oc1ccccc1C(=O)O"
asp_score = dock(ASPIRIN, "aspirin")
print(f"Aspirin docking score: {asp_score:.2f} kcal/mol")

## 7. Which amino acids does aspirin touch?
Residues with any atom within 4 Å of the docked aspirin.

In [ ]:
prot_xyz = xyz(protein)
labels = [(l[17:20].strip(), int(l[22:26])) for l in protein]
asp_xyz = read_xyz_pdb("aspirin_best.pdb")

d = np.linalg.norm(prot_xyz[:, None, :] - asp_xyz[None, :, :], axis=2).min(axis=1)
near = sorted({labels[i] for i in np.where(d < 4.0)[0]}, key=lambda x: x[1])
print("Residues within 4 Å:", ", ".join(f"{n}{i}" for n, i in near))

known = {"ARG120", "TYR355", "SER530", "TYR385"}
found = {f"{n}{i}" for n, i in near}
print("Known key residues also contacted:", sorted(known & found) or "none")

Literature highlights **Arg120** and **Tyr355** at the channel entrance and **Ser530**, the residue aspirin chemically acetylates. Docking predicts non-covalent placement only, so it cannot show the acetylation itself, but contact with these residues is a good sign.

## 8. Compare several remedies

In [ ]:
drugs = {
    "Aspirin": ASPIRIN,
    "Ibuprofen": IBU,
    "Paracetamol": "CC(=O)Nc1ccc(O)cc1",
    "Naproxen": "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
    "Caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
}
rows = []
for name, smi in drugs.items():
    rows.append({"Drug": name, "Score (kcal/mol)": round(dock(smi, name.lower()), 2)})
df = pd.DataFrame(rows).sort_values("Score (kcal/mol)")
df

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.barh(df["Drug"], df["Score (kcal/mol)"], color="#4C78A8")
plt.xlabel("Docking score (kcal/mol), more negative = stronger")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Reading the results:** NSAIDs like ibuprofen and naproxen are known COX binders, so they should score well. Caffeine is not a COX inhibitor and acts as a rough negative control. Paracetamol works differently from aspirin. Docking scores are approximate (typically off by 1-2 kcal/mol), so treat the ranking as a hint, not proof.

## 9. 3D view
Gray: COX-1 · green: docked aspirin · magenta: real ibuprofen from the crystal · sticks: nearby residues.

In [ ]:
view = py3Dmol.view(width=600, height=500)
view.addModel("\n".join(protein), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray", "opacity": 0.6}})
view.addStyle({"model": 0, "resi": [i for _, i in near]}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.15}})

view.addModel(open("aspirin_best.pdb").read(), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

view.addModel("\n".join(ibp), "pdb")
view.setStyle({"model": 2}, {"stick": {"colorscheme": "magentaCarbon"}})

view.zoomTo({"model": 1})
view.show()

## Limitations
- Rigid receptor: the protein does not move during docking.
- Heme and water were removed.
- Scores are estimates; real binding needs lab measurements.
- Aspirin binds irreversibly (covalent acetylation), which standard docking does not model.

## Next steps
- Add flexible side chains or covalent docking
- Run molecular dynamics (OpenMM) on the best pose
- Dock against COX-2 and compare selectivity